In [ ]:
from itertools import pairwise
import ray
import torch
import numpy as np

In [ ]:
# Interesting helper functions

GPU_FRACTION=0.02

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

# @ray.remote(num_gpus=GPU_FRACTION)
def rand_matmul(low, high, r):
    # print(f"rand_mat: {n=}, {m=}, {r=}")
    k, k = torch.randint(low, high, (2,))
    A = torch.randn(high, k, r).to(device)
    B = torch.randn(high, k, r).to(device)
    # print(f"rand_matmul: [{list(A.shape)}] @ [{list(B.shape)=}]")
    return torch.einsum("lkr,hkr->lh", A, B)

# @ray.remote(num_gpus=GPU_FRACTION*4)
def bridge_tensor(a, c):
    """returns a matrix b of the smallest shape so that: a @ b @ c is valid"""
    # print(f"bridge_tensor: {a=}, {c=}")
    return torch.randn(a.shape[-1], c.shape[0]).to(device)

@ray.remote(num_gpus=GPU_FRACTION)
def mul_chain(low=10_000, high=20_000, r=1_000, l=2) -> torch.Tensor | None:
    # print(f"mul_chain: {m=}, {n=}, {r=}, {l=}")
    assert l >= 2
    # tensors = ray.get([rand_matmul.remote(low=low, high=high, r=r) for _ in range(l)])
    # tensors = [rand_matmul(low=low, high=high, r=r) for _ in range(l)]

    # print(f"{tensors=}")
    
    # bridge_tensors = ray.get([bridge_tensor.remote(a, c) for a, c in pairwise(tensors)])
    # bridge_tensors = [bridge_tensor(a, c) for a, c in pairwise(tensors)]

    # print(f"{bridge_tensors=}")
    # assert len(bridge_tensors) == (len(tensors) - 1)
    res = torch.ones(high, high).to(device)
    for _ in range(l):
        a, c = [rand_matmul(low=low, high=high, r=r) for _ in range(2)]
        b = bridge_tensor(a, c)
        # for (a, c), b in zip(pairwise(tensors), bridge_tensors):
        # print(f"attempting: {a=} @ {b=} @ {c=}")
        # print(f"attempting: {a.shape=} @ {b.shape=} @ {c.shape=}")
        for _ in range(10):
            res += a @ b @ c @ res
        # print(f"\tresult: {d.shape}")
        # print(f"mul_chain: {list(d.shape)} = {list(a.shape)} @ {list(b.shape)} @ {list(c.shape)}")
        
    return res

In [ ]:
# Make the GPUs sweat
mc_ftrs = [mul_chain.remote(low=100, high=800, r=128, l=32) for _ in range(1)]
mcs = ray.get(mc_ftrs)
print([mc.shape for mc in mcs])

In [ ]:
# (0, 1, 2)
a = torch.randn(2, 3, 4)
print(a)

In [ ]:
a[0, 0, 2]

In [ ]:
ap = a.permute(2, 0, 1)
print(ap.shape)
print(ap)

In [ ]:
futures = [matmul_gpu.remote() for i in range(1000)]

results = ray.get(futures)
for r in results:
    print(r)

In [ ]:
from ray.train import ScalingConfig
from ray.train.torch import train_loop_utils, TorchTrainer
from torchvision.datasets import CIFAR10
from torchvision.transforms import ToTensor

scaling_config = ScalingConfig(
    num_workers=2,
    use_gpu=True
)

In [ ]:
# # 1. Download/load the dataset using torchvision
# torch_dataset = CIFAR10(root="/home/nico/ray-test/notebooks/data", train=True, download=True, transform=ToTensor())

# # 2. Convert to Ray Data
# ray_dataset = ray.data.from_torch(torch_dataset)

# print(ray_dataset.count())